# 01 — Train the 1D CNN

**Purpose:** train a CREB1 binding classifier on the dataset produced by `00_data_sanity.ipynb`.

This notebook is a **thin consumer** of `src/`. All training logic lives in `src/train.py`, so the same code can be run from the terminal (`python -m src.train`) or in this notebook. If you find yourself writing a training loop here, put it in `src/train.py` instead.

**Model:** `Creb1CNN` from `src/model.py` — a 3-block 1D CNN with adaptive average pooling, producing a single logit per sequence.

**Training config:**
- Loss: `BCEWithLogitsLoss` (sigmoid + BCE, numerically stable)
- Optimizer: Adam, lr = 1e-3
- Batch size: 64
- Epochs: 20
- Split: test chromosomes `1, 2`; validation is a random 10% of the training rows (this is for monitoring only — the leak-free split is train↔test).

**Outputs:** `checkpoints/cnn.pt` (best-val-loss checkpoint) and an in-memory loss history.

In [1]:
from src.notebook_setup import set_repo_root
set_repo_root()

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data import DataConfig, build_dataset, split_by_chromosome
from src.model import Creb1CNN
from src.train import make_loaders, train, save_checkpoint

## 1. Configuration

All hyperparameters are plain variables here, edit and re-run the training cell without touching `src/`.

**Note on `test_chroms`:** chromosome-level holdout is the standard defense against near-duplicate leakage in genomics models. `("1", "2")` follows the convention used by Basenji, Enformer, and most TF-binding benchmarks.

**Note on the validation split:** validation is a *random row split* within the training chromosomes, not a chromosome holdout. This is intentional — val loss is for monitoring overfitting during training, not for reporting final numbers. The leak-free number is the test AUROC computed in `02_eval.ipynb`.

In [3]:
cfg = DataConfig(window=500, neg_per_pos=1, seed=0, test_chroms=("1", "2"))
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = "checkpoints/cnn.pt"

print(f"device: {DEVICE}")
print(f"config: {cfg}")

device: cpu
config: DataConfig(bed_path=PosixPath('data/raw/creb1_peaks.bed'), fasta_path=PosixPath('data/raw/Homo_sapiens.GRCh38.dna.primary_assembly.fa'), out_dir=PosixPath('data/processed'), window=500, neg_per_pos=1, seed=0, test_chroms=('1', '2'))


## 2. Load the cached dataset

`build_dataset` loads from `data/processed/*.npz` if the cache exists. This should take under a second — if it takes minutes, the cache is missing and the pipeline is rebuilding from the FASTA.

The three-way split below is:
- **train / val** — both from training chromosomes (`3`–`22`, `X`, `Y`), val is a random 10% of rows
- **test** — held out chromosomes `1` and `2`, used only in `02_eval.ipynb`

**What to check:** shapes are `(N, 4, 500)` and positive rates are ≈ 0.5 across all three splits.

In [4]:
data = build_dataset(cfg)
split = split_by_chromosome(data, test_chroms=cfg.test_chroms)

rng = np.random.default_rng(cfg.seed)
val_mask = rng.random(len(split["y_train"])) < 0.1

X_tr = split["X_train"][~val_mask]; y_tr = split["y_train"][~val_mask]
X_va = split["X_train"][val_mask];  y_va = split["y_train"][val_mask]
X_te = split["X_test"]; y_te = split["y_test"]

print(f"train: {X_tr.shape}  val: {X_va.shape}  test: {X_te.shape}")
print(f"pos rate — train {y_tr.mean():.3f}  val {y_va.mean():.3f}  test {y_te.mean():.3f}")

train: (68054, 4, 500)  val: (7671, 4, 500)  test: (15725, 4, 500)
pos rate — train 0.491  val 0.496  test 0.540
